# Built Different? What the Data Really Says About Winning in the UFC

*Every UFC fan has a theory. "It's all about the reach." "Heavyweights just need one punch." "You can't teach size." But what happens when we stop debating and start counting? We analysed over 2,200 UFC fights to find out what actually separates winners from losers — and the answers might surprise you.*

## Introduction

The UFC is the world's biggest mixed martial arts organisation, where fighters from every discipline — boxing, wrestling, jiu-jitsu, muay thai — compete against each other in an eight-sided cage. With weight classes ranging from 52 kg strawweights to 120 kg heavyweights, the sport creates a natural laboratory for a question that has fascinated fight fans for decades: **what actually predicts who wins?**

Common wisdom says taller fighters have a reach advantage, heavier fighters hit harder, and younger fighters have the edge in athleticism. But conventional wisdom is often wrong. In this post, we use data from 2,228 UFC fights (spanning 2014–2018) to test these assumptions. We'll look at how fights end across different weight classes, whether being taller genuinely helps, and what statistics truly separate elite fighters from the rest. Finally, we'll build a statistical model to predict fight outcomes — and examine the fighters who defy the numbers.

Let's step into the Octagon.

## The Data

The dataset comes from a comprehensive UFC statistics collection on Kaggle, originally covering 2,318 fights from February 2014 to December 2018. After cleaning — removing draws, no-contests, and fights with missing outcomes, and aggregating round-by-round statistics into fight totals — we work with **2,228 decisive fights**. Of these, 1,837 have complete round-by-round performance statistics (strikes, takedowns, control time, and so on).

Each fight record contains both fighters' physical attributes (height, weight, age), their fight outcomes (winner, method of victory), and detailed performance metrics: significant strikes landed and attempted, takedowns, knockdowns, submission attempts, and control time. The cleaning script also creates derived "difference" features — e.g. `height_diff`, `sig_strike_diff` — that compare the two fighters directly in each bout.

One note: the original dataset does not include reach measurements, so we use **height** (and height difference) as a proxy for reach. Height and reach are strongly correlated in fighters, so this is a reasonable substitution.

In [1]:
# ============================================================
# SECTION 2: Data Exploration
# ============================================================
import pandas as pd

# Load the raw dataset
df = pd.read_csv('data/raw/ufcdataset.csv')

# --- How big is the dataset? ---
print(f"Rows: {df.shape[0]}")        # 2,318 fights
print(f"Columns: {df.shape[1]}")     # 894 columns (lots of round-by-round detail)

# --- What are the key (non-round) columns? ---
key_cols = [c for c in df.columns if '_Round' not in c]
print(f"\nKey columns ({len(key_cols)}):")
for c in key_cols:
    print(f"  {c}")

# --- First 5 rows (key columns only) ---
df[key_cols].head()

Rows: 2318
Columns: 894

Key columns (24):
  BPrev
  BStreak
  B_Age
  B_Height
  B_HomeTown
  B_ID
  B_Location
  B_Name
  B_Weight
  Date
  Event_ID
  Fight_ID
  Last_round
  Max_round
  RPrev
  R_Age
  R_Height
  R_HomeTown
  R_ID
  R_Location
  R_Name
  R_Weight
  winby
  winner


,BPrev,BStreak,B_Age,B_Height,B_HomeTown,B_ID,B_Location,B_Name,B_Weight,Date,...,RPrev,R_Age,R_Height,R_HomeTown,R_ID,R_Location,R_Name,R_Weight,winby,winner
0,0,0,38.0,193.0,Hounslow England,808,Amsterdam The Netherlands,Alistair Overeem,120.0,02/03/2014,...,0,39.0,190.0,"Las Vegas, Nevada USA",377,"Las Vegas, Nevada USA",Frank Mir,119.0,DEC,blue
1,0,0,36.0,172.0,"Chicago, Illinois United States",1054,"Chicago, Illinois United States",Ricardo Lamas,65.0,02/03/2014,...,0,32.0,170.0,Manaus Brazil,1052,Rio de Janeiro Brazil,Jose Aldo,65.0,DEC,red
2,0,0,39.0,167.0,"Isla Vista , California USA",959,"Sacramento, California USA",Urijah Faber,61.0,02/03/2014,...,0,31.0,167.0,Natal Brazil,1527,Rio de Janeiro Brazil,Renan Barao,61.0,KO/TKO,red
3,0,0,33.0,167.0,"San Diego, CA USA",1056,"San Diego, CA USA",Danny Martinez,56.0,02/03/2014,...,0,37.0,160.0,"San Jose, California USA",1253,"Tucson, Arizona USA",Chris Cariaso,56.0,DEC,red
4,0,0,36.0,185.0,Southampton England,2005,Southampton England,Tom Watson,84.0,02/03/2014,...,0,37.0,182.0,"Englewood, NJ USA",464,"Brick, NJ USA",Nick Catone,84.0,DEC,red


In [2]:
# --- Missing values for key columns ---
missing = df[key_cols].isnull().sum()
missing[missing > 0]

B_Age         17
B_Height      17
B_HomeTown    17
B_Location    13
B_Weight      12
R_Age         26
R_Height      24
R_HomeTown    25
R_Location    24
R_Weight      22
winby         36
dtype: int64

In [3]:
# --- Data types ---
df[key_cols].dtypes

BPrev           int64
BStreak         int64
B_Age         float64
B_Height      float64
B_HomeTown     object
B_ID            int64
B_Location     object
B_Name         object
B_Weight      float64
Date           object
Event_ID        int64
Fight_ID        int64
Last_round      int64
Max_round       int64
RPrev           int64
R_Age         float64
R_Height      float64
R_HomeTown     object
R_ID            int64
R_Location     object
R_Name         object
R_Weight      float64
winby          object
winner         object
dtype: object

In [4]:
# --- Fight outcome distributions ---
print("Winner (corner colour):")
print(df['winner'].value_counts())
print("\nWin method:")
print(df['winby'].value_counts())

Winner (corner colour):
winner
red           1327
blue           951
no contest      24
draw            16
Name: count, dtype: int64

Win method:
winby
DEC       1111
KO/TKO     744
SUB        427
Name: count, dtype: int64


In [5]:
# --- Weight class mapping ---
# Weights are in kg — these map to standard UFC divisions
weight_map = {
    52: 'Strawweight', 56: 'Flyweight', 61: 'Bantamweight',
    65: 'Featherweight', 70: 'Lightweight', 77: 'Welterweight',
    84: 'Middleweight', 93: 'Light Heavyweight', 120: 'Heavyweight'
}
df['weight_class'] = df['B_Weight'].map(
    lambda x: weight_map.get(int(x)) if pd.notna(x) and int(x) in weight_map else 'Other'
)
print(df['weight_class'].value_counts())

weight_class
Welterweight         417
Lightweight          407
Bantamweight         306
Middleweight         259
Featherweight        251
Flyweight            209
Light Heavyweight    188
Other                124
Strawweight          117
Heavyweight           40
Name: count, dtype: int64


## Setup for the Analysis

Before plotting anything, we load the cleaned dataset and set up a consistent visual style that will be used across every chart in this post. Consistent colours make a blog post feel professional — the same three colours will represent KO/TKO, Submission, and Decision throughout.

In [ ]:
# ── Imports and load cleaned data ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Consistent plot style across the whole notebook
plt.style.use('seaborn-v0_8-whitegrid')

# Consistent colour palette used in every plot
COLORS = {
    'primary':   '#E63946',  # Red — KO/TKO, emphasis
    'secondary': '#457B9D',  # Blue — Submission / secondary
    'tertiary':  '#2A9D8F',  # Teal — Decision
    'accent':    '#F4A261',  # Amber — highlights
    'dark':      '#1D3557',  # Dark navy — text / headers
}
METHOD_COLORS = {
    'KO/TKO':     COLORS['primary'],
    'Submission': COLORS['secondary'],
    'Decision':   COLORS['tertiary'],
}

# Ensure output folder exists for saved figures
os.makedirs('output/figures', exist_ok=True)

# Load the cleaned dataset produced by Section 3
df = pd.read_csv('data/cleaned/ufc_cleaned.csv')
print(f"Cleaned dataset loaded: {df.shape[0]} fights, {df.shape[1]} columns")
print(f"Fights with detailed round stats: {(df['r_sig_strikes_attempted'] > 0).sum()}")

# Display order for weight classes (lightest to heaviest)
wc_order = ['Strawweight', 'Flyweight', 'Bantamweight', 'Featherweight',
            'Lightweight', 'Welterweight', 'Middleweight', 'Light Heavyweight', 'Heavyweight']

## How Do Fights End? The Weight Class Effect

If you've ever watched a heavyweight fight, you know the feeling — every exchange carries the threat of sudden ending. But just how dramatic is the difference between weight classes when it comes to *how* fights are decided? We broke down the proportion of KO/TKO, submission, and decision wins across all nine UFC weight classes.

In [ ]:
# ── Plot 1: Win Methods by Weight Class (Stacked Bar Chart) ──
fig, ax = plt.subplots(figsize=(12, 6))

# Proportion of each win method within each weight class
wc_method = df.groupby(['weight_class', 'win_method']).size().unstack(fill_value=0)
wc_method = wc_method.reindex(wc_order)
wc_method_pct = wc_method.div(wc_method.sum(axis=1), axis=0) * 100

# Build stacked bars: KO/TKO on the bottom, Submission in the middle, Decision on top
methods = ['KO/TKO', 'Submission', 'Decision']
bottom = np.zeros(len(wc_order))
for method in methods:
    vals = wc_method_pct[method].values
    ax.bar(wc_order, vals, bottom=bottom, color=METHOD_COLORS[method],
           label=method, edgecolor='white', linewidth=0.5)
    # Label segments larger than 20%
    for i, v in enumerate(vals):
        if v > 20:
            ax.text(i, bottom[i] + v / 2, f'{v:.0f}%', ha='center', va='center',
                    fontsize=8, fontweight='bold', color='white')
    bottom += vals

ax.set_xlabel('Weight Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Proportion of Wins (%)', fontsize=12, fontweight='bold')
ax.set_title('How Fights End Across Weight Classes', fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Win Method', loc='upper right', framealpha=0.9)
ax.set_xticklabels(wc_order, rotation=30, ha='right')
plt.tight_layout()
plt.savefig('output/figures/plot1_win_methods_by_weight.png', dpi=150, bbox_inches='tight')
plt.show()

The pattern is striking — and it runs exactly in the direction you'd expect if you're a fight fan, but with some twists. At **heavyweight, around 54% of fights end in KO/TKO**, while only a third go to the judges. Compare that with **strawweight, where roughly 69% of fights go to decision** and barely 9% end in knockout. The trend is remarkably consistent: as fighters get bigger, the knockout rate climbs steadily.

But the submission story is more interesting. Submissions peak at the **lighter weight classes** — flyweight and bantamweight — where around 23% of fights end on the mat. This makes intuitive sense: lighter fighters tend to be more agile on the ground and can more easily execute the sweeps and transitions that lead to chokes and joint locks. By the time you reach heavyweight, submissions account for only about 10% of victories.

What this tells us is that the UFC isn't one sport — it's really several sports wearing the same logo. A flyweight contest is fundamentally a different kind of athletic event from a heavyweight bout, and the data bears this out. If you're betting on a heavyweight fight, history says there's a coin-flip chance someone's getting knocked out. At strawweight, settle in — you're probably going the distance.

## Does Height Actually Help?

One of the oldest debates in combat sports is whether being taller gives a fighter a meaningful advantage. The theory is simple: taller fighters can hit their opponents from further away while staying out of range. But fighting isn't boxing — in MMA, a taller fighter can be taken down, clinched up, or find their length neutralised by an aggressive shorter opponent. So what does the data say?

We measured the height difference between the two fighters in each fight, grouped them into 5 cm buckets, and calculated the win rate at each height gap.

In [ ]:
# ── Plot 2: Height Advantage vs Win Rate (Bubble Scatter + Trend) ──
fig, ax = plt.subplots(figsize=(10, 6))

plot_data = df[df['height_diff'].notna()].copy()

# Bin into 5 cm buckets for a cleaner visualisation
bins = np.arange(-25, 27.5, 5)
plot_data['height_bin'] = pd.cut(plot_data['height_diff'], bins=bins)
binned = plot_data.groupby('height_bin', observed=True).agg(
    win_rate=('r_win', 'mean'),
    count=('r_win', 'count'),
    height_mid=('height_diff', 'mean'),
).dropna()
binned = binned[binned['count'] >= 10]  # require at least 10 fights per bin

# Bubble size scales with the number of fights in each bucket
ax.scatter(binned['height_mid'], binned['win_rate'] * 100,
           s=binned['count'] * 3, c=COLORS['dark'], alpha=0.7,
           zorder=5, edgecolors='white', linewidth=1)

# Fit a simple linear regression through the raw (unbinned) data
slope, intercept, r_value, p_value, _ = stats.linregress(
    plot_data['height_diff'], plot_data['r_win'] * 100)
x_line = np.linspace(-25, 25, 100)
ax.plot(x_line, slope * x_line + intercept, color=COLORS['primary'],
        linewidth=2.5, label=f'Trend line (slope={slope:.2f})')

ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% (no advantage)')
ax.set_xlabel('Height Advantage — Red Corner (cm)', fontsize=12, fontweight='bold')
ax.set_ylabel('Red Corner Win Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Does Being Taller Actually Help You Win?', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right', framealpha=0.9)
ax.set_xlim(-27, 27)
ax.set_ylim(30, 80)
ax.annotate('Bubble size = number of fights', xy=(0.02, 0.97),
            xycoords='axes fraction', fontsize=9, fontstyle='italic',
            va='top', color='gray')
plt.tight_layout()
plt.savefig('output/figures/plot2_height_advantage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Regression: slope={slope:.3f}, p-value={p_value:.3f}, R²={r_value**2:.4f}")

Here's where conventional wisdom takes a hit. The trend line slopes gently upward — taller fighters do win *slightly* more often — but the effect is **tiny and not statistically significant** (p ≈ 0.24). Fighters with a 5+ cm height advantage win about 61% of the time, compared to roughly 58% for those giving up 5+ cm. That small gap is barely distinguishable from no effect at all.

In other words, **height barely matters**. This is a great example of why data analysis is valuable: the "obvious" advantage that every casual fan assumes exists turns out to be nearly invisible in the numbers. MMA is simply too multidimensional. A shorter fighter with elite wrestling or a devastating overhand can completely neutralise a height advantage. Think of fighters like Daniel Cormier or Henry Cejudo — both undersized for their divisions, both dominant champions.

The takeaway: if you're picking a fight-night winner, the tale of the tape isn't going to help you much. You need to look deeper.

## Winners vs Losers: The Numbers at a Glance

So if height doesn't matter much, what does? We compared the average statistics of fight winners against fight losers across every fight with detailed stats. The table below shows the mean and standard deviation for each group, plus the percentage difference between them.

In [ ]:
# ── Plot 3: Summary Statistics Table — Winners vs Losers ──

# Restrict to fights with detailed round-level stats
with_stats = df[df['r_sig_strikes_attempted'] > 0].copy()

# Build parallel lists of winner-stats and loser-stats
# If r_win == 1, the red corner won; otherwise the blue corner won
winners_list, losers_list = [], []
for _, row in with_stats.iterrows():
    w_p = 'r' if row['r_win'] == 1 else 'b'   # winner's prefix
    l_p = 'b' if row['r_win'] == 1 else 'r'   # loser's prefix
    winners_list.append({
        'Height (cm)':           row[f'{w_p}_height'],
        'Age':                   row[f'{w_p}_age'],
        'Sig. Strikes Landed':   row[f'{w_p}_sig_strikes_landed'],
        'Striking Accuracy (%)': row[f'{w_p}_sig_strike_accuracy'],
        'Takedowns Landed':      row[f'{w_p}_takedowns_landed'],
        'Knockdowns':            row[f'{w_p}_knockdowns'],
    })
    losers_list.append({
        'Height (cm)':           row[f'{l_p}_height'],
        'Age':                   row[f'{l_p}_age'],
        'Sig. Strikes Landed':   row[f'{l_p}_sig_strikes_landed'],
        'Striking Accuracy (%)': row[f'{l_p}_sig_strike_accuracy'],
        'Takedowns Landed':      row[f'{l_p}_takedowns_landed'],
        'Knockdowns':            row[f'{l_p}_knockdowns'],
    })

win_df  = pd.DataFrame(winners_list)
lose_df = pd.DataFrame(losers_list)

# Build the comparison table
metrics = ['Height (cm)', 'Age', 'Sig. Strikes Landed',
           'Striking Accuracy (%)', 'Takedowns Landed', 'Knockdowns']
table_rows = []
for m in metrics:
    wm, ws = win_df[m].mean(),  win_df[m].std()
    lm, ls = lose_df[m].mean(), lose_df[m].std()
    diff   = ((wm - lm) / lm * 100) if lm != 0 else 0
    table_rows.append([m, f'{wm:.1f}', f'{ws:.1f}', f'{lm:.1f}', f'{ls:.1f}', f'{diff:+.1f}%'])

# Render the table with matplotlib
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')
col_labels = ['Metric', 'Winners\n(Mean)', 'Winners\n(SD)',
              'Losers\n(Mean)', 'Losers\n(SD)', 'Difference']
table = ax.table(cellText=table_rows, colLabels=col_labels, loc='center',
                 cellLoc='center', colColours=[COLORS['dark']] * 6)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

# Style the header row
for j in range(6):
    table[0, j].set_text_props(color='white', fontweight='bold')
    table[0, j].set_facecolor(COLORS['dark'])

# Alternate row shading + highlight the difference column
for i in range(1, len(table_rows) + 1):
    for j in range(6):
        cell = table[i, j]
        cell.set_facecolor('#f0f4f8' if i % 2 == 0 else 'white')
        if j == 5:
            cell.set_text_props(fontweight='bold', color=COLORS['primary'])

ax.set_title('Winners vs Losers: Key Statistics Compared',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('output/figures/plot3_summary_table.png', dpi=150, bbox_inches='tight')
plt.show()

The numbers confirm what we saw in the height plot: **physical attributes barely differ** between winners and losers. Height is essentially identical (a difference of less than 1%), and winners are only slightly *younger* on average — suggesting that youth provides a small edge, but nothing dramatic.

Where the real gaps appear is in **performance stats**. Winners land around 14% more significant strikes than losers and complete roughly 20% more takedowns. Striking accuracy shows only a small difference, but volume matters far more: winners don't necessarily land a higher *proportion* of their shots, they simply throw and land *more*.

This points to something important. The fighters who win in the UFC tend to be the ones who **impose their game plan more aggressively**, not simply those who are physically bigger or marginally more accurate. Keep that in mind when we scale this observation up in the next plot.

## What Separates the Best from the Rest?

Average winners are one thing — but what about the *elite*? We identified every fighter in the dataset with at least 5 fights and a 70%+ win rate (the "elite" group), and compared their average performance against fighters who had 3+ fights but won fewer than half of them (the "below average" group). The chart below shows how elite fighters perform relative to the below-average group (set as a baseline of 100%).

In [ ]:
# ── Plot 4: Elite Fighters vs Below-Average (Grouped Bar Chart, Normalised) ──

# Build career-level stats: each fighter's average across all their appearances.
# Since each fight has a red-corner fighter AND a blue-corner fighter, each fight
# contributes TWO records to the fighter-level dataset.
with_stats = df[df['r_sig_strikes_attempted'] > 0].copy()
records = []
for _, row in with_stats.iterrows():
    # Red-corner fighter's record
    records.append({
        'name':           row['r_name'],
        'won':            int(row['r_win'] == 1),
        'sig_str_landed': row['r_sig_strikes_landed'],
        'sig_str_att':    row['r_sig_strikes_attempted'],
        'td_landed':      row['r_takedowns_landed'],
        'kd':             row['r_knockdowns'],
    })
    # Blue-corner fighter's record
    records.append({
        'name':           row['b_name'],
        'won':            int(row['r_win'] == 0),
        'sig_str_landed': row['b_sig_strikes_landed'],
        'sig_str_att':    row['b_sig_strikes_attempted'],
        'td_landed':      row['b_takedowns_landed'],
        'kd':             row['b_knockdowns'],
    })

fighter_df = pd.DataFrame(records)
career = fighter_df.groupby('name').agg(
    total_fights    = ('won', 'count'),
    wins            = ('won', 'sum'),
    avg_sig_str     = ('sig_str_landed', 'mean'),
    avg_sig_str_att = ('sig_str_att', 'mean'),
    avg_td          = ('td_landed', 'mean'),
    avg_kd          = ('kd', 'mean'),
).reset_index()
career['win_rate']     = career['wins']        / career['total_fights']    * 100
career['str_accuracy'] = career['avg_sig_str'] / career['avg_sig_str_att'] * 100

# Define the two groups
elite = career[(career['total_fights'] >= 5) & (career['win_rate'] >= 70)]
rest  = career[(career['total_fights'] >= 3) & (career['win_rate'] <  50)]

# Metrics to compare
compare = {
    'Sig. Strikes\nper Fight':  (elite['avg_sig_str'].mean(),  rest['avg_sig_str'].mean()),
    'Striking\nAccuracy (%)':   (elite['str_accuracy'].mean(), rest['str_accuracy'].mean()),
    'Takedowns\nper Fight':     (elite['avg_td'].mean(),       rest['avg_td'].mean()),
    'Knockdowns\nper Fight':    (elite['avg_kd'].mean(),       rest['avg_kd'].mean()),
    'Win Rate\n(%)':            (elite['win_rate'].mean(),     rest['win_rate'].mean()),
}

# Normalise so the below-average group = 100% on every metric
labels     = list(compare.keys())
elite_norm = [e / r * 100 for e, r in compare.values()]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width / 2, elite_norm, width,
               label=f'Elite Fighters (n={len(elite)})',
               color=COLORS['primary'], edgecolor='white', linewidth=0.5)
ax.bar(x + width / 2, [100] * len(labels), width,
       label=f'Below Average (n={len(rest)})',
       color=COLORS['secondary'], edgecolor='white', linewidth=0.5)

# Label the elite bars with their percentage values
for bar, val in zip(bars1, elite_norm):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{val:.0f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color=COLORS['primary'])

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.4)
ax.set_ylabel('Relative Performance\n(Below Average = 100%)',
              fontsize=11, fontweight='bold')
ax.set_title('What Separates the Best from the Rest?',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(0, max(elite_norm) * 1.15)
plt.tight_layout()
plt.savefig('output/figures/plot4_elite_vs_average.png', dpi=150, bbox_inches='tight')
plt.show()

The differences here are massive. Elite fighters land **more than twice as many significant strikes** per fight, complete **over twice as many takedowns**, and score **over twice as many knockdowns**. But look at the outlier: **striking accuracy is barely different**. This reinforces exactly what we saw in the summary table — elite fighters don't win because they're more *precise*, they win because they're more *active* and *dominant*.

If you're looking for the statistical fingerprint of a great UFC fighter, don't look at their accuracy percentage. Look at their **volume** — how many strikes they land, how many takedowns they complete, how many times they put their opponent on the canvas. The best fighters in the sport win by sheer relentless activity, not by being marginally more accurate with their shots.

So far we've established the "what": volume and finishing ability separate winners from losers far more than physical attributes do. In the next section, we formalise this with a logistic regression model that predicts fight outcomes — and we'll also identify the fighters who defy the numbers entirely.

---

*Sections 5 & 6 (regression model and outlier analysis) will follow below.*

---